# End-to-end walkthrough — Module A pipeline

**Objective:** Trace a single **entity** row from synthetic generation through every pipeline stage:  
generation → flaw injection → cleaning → feature engineering → segmentation → propensity scoring.

All RNG seeds fixed to **42** for full reproducibility.

| Stage | Module | Output shown |
|-------|--------|--------------|
| 1. Generate | `data.generator` | `entity_id`, age, department, `has_cedula` |
| 2. Inject flaws | `data.raw_injector` | duplicate count, null delta |
| 3. Clean | `data.cleaner` | 14-step QA, clean row |
| 4. Features | `features.*` | behavioral + demographic + reachability |
| 5. Segment | `models.segmentation` | `segment_label`, silhouette |
| 6. Propensity | `models.propensity` | `participation_propensity`, AUC-ROC |

In [ ]:
from __future__ import annotations

from pathlib import Path

import yaml

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").is_file(), "Run from repo root: cd <repo_root> && jupyter notebook"

with open(ROOT / "module_a_population_segmentation/config/generation.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
with open(ROOT / "module_a_population_segmentation/config/calibration_anchors.yaml", encoding="utf-8") as f:
    anchors = yaml.safe_load(f)

SAMPLE_SIZE = 15_000
config["sample_size"] = SAMPLE_SIZE
print(f"Config loaded. sample_size={SAMPLE_SIZE}, departments={len(config['department_weights'])}")

## Step 1 — Generate synthetic population

`generate_population` creates N entities from the YAML distribution spec.  
Values mirror 2018 census department weights — no real PII.

In [ ]:
from population_segmentation.data.generator import generate_population

raw = generate_population(config, seed=42)
print(f"Generated {len(raw):,} entities, {raw.shape[1]} columns")

ENTITY_IDX = 0
entity_id = int(raw["entity_id"].iloc[ENTITY_IDX])
print(f"\n--- Entity #{entity_id} (raw, before any flaws) ---")
display(raw[raw["entity_id"] == entity_id][["entity_id", "age", "sex", "department", "has_cedula", "is_rural", "education_level"]].T)

## Step 2 — Inject realistic data flaws

`inject_flaws` simulates survey-quality issues: duplicate rows, null fields, cedula format errors.  
Flaw rates are controlled by `config['flaw_rates']` in `generation.yaml`.

In [ ]:
from population_segmentation.data.raw_injector import inject_flaws

raw_dirty = inject_flaws(raw, config, seed=42)

print(f"Rows:  {len(raw):,} → {len(raw_dirty):,}  (+{len(raw_dirty) - len(raw)} duplicate rows injected)")
print(f"Nulls: {raw.isnull().sum().sum():,} → {raw_dirty.isnull().sum().sum():,}")

dirty_rows = raw_dirty[raw_dirty["entity_id"] == entity_id]
print(f"\n--- Entity #{entity_id} appears {len(dirty_rows)} time(s) in dirty frame ---")
display(dirty_rows[["entity_id", "age", "sex", "department", "has_cedula"]].T)

## Step 3 — 14-step cleaning pipeline

`clean_population` runs 14 sequential QA gates: dedup, null imputation, cedula normalisation,  
age clamping, department validation, outlier flags, and more.  
A QA report is written to disk at each run.

In [ ]:
import tempfile
from pathlib import Path

from population_segmentation.data.cleaner import clean_population

qa_dir = Path(tempfile.mkdtemp())
clean_df = clean_population(raw_dirty, config, qa_report_dir=qa_dir, seed=42)

print(f"Rows after cleaning: {len(clean_df):,}  (removed {len(raw_dirty) - len(clean_df):,} rows)")
print(f"Null cells remaining: {clean_df.isnull().sum().sum()}")

clean_row = clean_df[clean_df["entity_id"] == entity_id]
if len(clean_row):
    print(f"\n--- Entity #{entity_id} after cleaning ---")
    display(clean_row[["entity_id", "age", "sex", "department", "has_cedula", "is_rural", "education_level"]].T)
else:
    print(f"Entity #{entity_id} removed by a QA gate — using next available entity.")
    entity_id = int(clean_df["entity_id"].iloc[0])
    display(clean_df[clean_df["entity_id"] == entity_id][["entity_id", "age", "sex", "department", "has_cedula"]].T)

## Step 4 — Feature engineering

Three layers stacked in sequence:
- **Demographic** — age group, dependency ratio, education index  
- **Behavioral** — structural dependency flag, jopará index  
- **Reachability** — TV/radio/WhatsApp penetration, internet access, primary reach channel

In [ ]:
from population_segmentation.features.behavioral import build_behavioral_features
from population_segmentation.features.demographic import build_demographic_features
from population_segmentation.features.reachability import build_reachability_features

feat_df = build_reachability_features(
    build_behavioral_features(build_demographic_features(clean_df))
)

new_cols = [c for c in feat_df.columns if c not in clean_df.columns]
print(f"Feature columns added: {len(new_cols)}")
print(f"  {new_cols}")

print(f"\n--- Entity #{entity_id} feature vector ---")
display(feat_df[feat_df["entity_id"] == entity_id][new_cols].T)

## Step 5 — Segmentation (k-means, k=6)

`build_segmentation_frame` runs DBSCAN noise filtering then k-means on PCA-reduced features.  
Returns human-readable segment labels (e.g. `high_reach_urban`, `rural_low_contact`)  
plus scalar diagnostics: silhouette score, bootstrap ARI, noise rate.

In [ ]:
from population_segmentation.models.segmentation import build_segmentation_frame

labels_df, seg_metrics = build_segmentation_frame(feat_df, k=6, random_state=42)

print("Segmentation diagnostics:")
for k, v in seg_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

print("\nSegment distribution:")
display(labels_df["segment_label"].value_counts().to_frame("count"))

seg_row = labels_df[labels_df["entity_id"] == entity_id]
print(f"\n--- Entity #{entity_id} segment assignment ---")
display(seg_row[["entity_id", "segment_label", "segment_id", "dbscan_noise_flag"]].T)

## Step 6 — Participation propensity (logistic regression)

`PropensityModel` trains a stratified logistic regression, then rakes department-level  
scores against census calibration anchors (`calibration_anchors.yaml`).  
Output: `participation_propensity` ∈ [0, 1].

In [ ]:
from typing import Any, cast

import yaml

from population_segmentation.models.propensity import PropensityModel

model_params_path = ROOT / "module_a_population_segmentation/config/model_params.yaml"
with open(model_params_path, encoding="utf-8") as f:
    model_params: dict[str, Any] = yaml.safe_load(f)
stratify_by = tuple(model_params["propensity"]["stratify_by"])

merged = feat_df.reset_index(drop=True).copy()
merged["segment_label"] = labels_df["segment_label"].to_numpy()
merged["segment_id"] = labels_df["segment_id"].to_numpy()
merged["dbscan_noise_flag"] = labels_df["dbscan_noise_flag"].to_numpy()

prop_raw = PropensityModel(random_state=42, stratify_by=stratify_by).fit_predict(merged, anchors)
prop_out = cast(dict[str, Any], prop_raw)
prop_metrics = prop_out["metrics"]

print(f"Model evaluation — AUC-ROC: {prop_metrics['auc_roc']:.4f}  Brier score: {prop_metrics['brier_score']:.4f}")

entity_idx = merged[merged["entity_id"] == entity_id].index[0]
propensity = float(prop_out["predictions"].iloc[entity_idx])
logit = float(prop_out["raw_logit_score"].iloc[entity_idx])
rake = float(prop_out["department_rake_multiplier"].iloc[entity_idx])
dept = merged.loc[entity_idx, "department"]
seg_label = merged.loc[entity_idx, "segment_label"]

print(f"\n--- Entity #{entity_id} final scores ---")
print(f"  department:               {dept}")
print(f"  segment:                  {seg_label}")
print(f"  raw logit score:          {logit:.4f}")
print(f"  department rake mult:     {rake:.4f}")
print(f"  participation_propensity: {propensity:.4f}")

## Summary — entity journey

| Stage | Key output |
|-------|------------|
| Raw generation | Census-calibrated attributes (no PII) |
| Flaw injection | Duplicates + nulls mimicking field survey quality |
| Cleaning | 14 QA gates — dedup, imputation, format normalisation |
| Features | Behavioral + reachability scores appended |
| Segmentation | `segment_label` from k-means (k=6) on PCA features |
| Propensity | `participation_propensity` ∈ [0,1] with department rake |

All seeds fixed to 42 — re-running produces identical outputs.